# 1 - Data, retrieval index and RAFT dataset

Part 1 of my RAFT vs RAG comparison on a single T4.

Dataset: Doc2Dial, customer-support style dialogues grounded in US government documents (SSA, VA, DMV, StudentAid). Why this one: every agent turn comes with the exact document span it was based on, so I get gold replies and gold grounding labels without a teacher LLM writing them.

What happens here, in order:
1. download Doc2Dial and chunk the documents into ~140-word passages (the RAG corpus)
2. build train / valid / test splits, including a test set on documents never seen in training
3. embed everything with bge-small and store the top-20 retrieved passages per query
4. build the RAFT training set: gold passage + hard-negative distractors, quote-then-answer target

Kaggle: Internet on. GPU optional (embedding takes ~1 min on a T4, CPU works too). Save & Run All at the end - notebooks 2 and 3 read this notebook's output as an input.

**The whole project in one picture**

The question: for a customer-support bot on a single cheap GPU, is it better to fine-tune a small model to *use* retrieved documents properly (RAFT), or to keep a stock model and just do RAG - maybe with a bigger model?

```
Doc2Dial docs -> passages -> bge-small index ---------------------------+
                                  |                                      |
Doc2Dial dialogues -> examples -> top-20 retrieved per query             |
                                  |                                      v
            (nb 1) RAFT set: gold + hard distractors, quote-then-answer   RAG at test time:
                                  |                                      top-k passages -> model
            (nb 2) QLoRA on Qwen2.5-1.5B -> merged RAFT model ----------> (nb 3) base 1.5B / RAFT 1.5B / 3B / 7B-AWQ
                                                                           -> F1, ROUGE-L, groundedness, judge
                                                                           -> latency, throughput, VRAM, $/1k queries
```

Every model sees the same retriever, the same passages and greedy decoding, so the only thing that changes is the model itself. Shared code (prompt, query, metrics) lives in `common.py`, which is written identically into all three notebooks.


In [ ]:
!pip -q install sentence-transformers pyyaml

In [ ]:
import os
os.makedirs('/kaggle/working/proj/configs', exist_ok=True)
os.makedirs('/kaggle/working/proj/src', exist_ok=True)
%cd /kaggle/working/proj

### Project files
The scripts are written out with `%%writefile`, so the notebook is self-contained and there's nothing to upload. `common.py` holds everything RAG and RAFT have to share (prompt template, retrieval query, metrics) so the comparison stays fair.

In [ ]:
%%writefile /kaggle/working/proj/configs/config.yaml
# RAFT vs RAG on a single T4. The same config is written in all three notebooks so every stage reads the same numbers.
seed: 42

data:
  doc2dial_url: https://doc2dial.github.io/file/doc2dial_v1.0.1.zip
  holdout_doc_frac: 0.10     # docs never seen in training -> "unseen-doc" test set
  passage_words: 140         # target passage size when chunking sections
  history_turns: 4           # previous turns shown in the prompt
  max_train: 8000
  max_valid: 200
  max_test_seen: 500
  max_test_unseen: 300
  valid_dialogue_frac: 0.04

retrieval:
  embed_model: BAAI/bge-small-en-v1.5
  query_prefix: "Represent this sentence for searching relevant passages: "
  save_top: 20               # stored per query (hard negatives + recall@k)
  k: 4                       # docs given to the model at test time

raft:
  n_docs: 4                  # docs in each training prompt
  p_oracle: 0.8              # gold passage is in the context 80% of the time; the RAFT paper found 60-100% all work
  hard_neg_frac: 0.7         # share of distractors taken from retriever top-10
  style: cot                 # cot = quote then answer | direct = answer only
  max_tokens: 1900           # drop longer examples so nothing gets truncated mid-answer at max_length 2048

train:
  base_model: Qwen/Qwen2.5-1.5B-Instruct     # stretch: Qwen/Qwen2.5-3B-Instruct
  max_length: 2048
  epochs: 2
  per_device_batch: 2        # x grad_accum 8 = effective batch 16. use 1 for the 3B model
  grad_accum: 8
  lr: 2.0e-4
  warmup_ratio: 0.03
  lora_r: 32
  lora_alpha: 64             # alpha = 2r, the usual starting point
  lora_dropout: 0.05
  save_steps: 200
  eval_steps: 200

eval:
  max_model_len: 4096
  gpu_util: 0.88             # share of VRAM vLLM grabs up front (weights + KV cache)
  max_new_tokens_direct: 256
  max_new_tokens_cot: 384

bench:
  n_queries: 200
  warmup: 10
  # rough on-demand T4 price, not a quote. cost per 1k queries is linear in it, so it's easy to redo
  gpu_usd_per_hour: 0.35

In [ ]:
%%writefile /kaggle/working/proj/src/common.py
"""Shared helpers: config, IO, prompts, answer parsing, metrics, retriever.

Everything that must be identical between the RAG baseline and the RAFT model
(prompt template, retrieval query, metrics) lives here so the comparison is fair.
"""
from __future__ import annotations

import json
import random
import re
import string
from collections import Counter
from pathlib import Path


# ----------------------------------------------------------------------------- IO
# read the yaml config into a plain dict - every script starts with this
def load_cfg(path: str = "configs/config.yaml") -> dict:
    import yaml

    with open(path) as f:
        return yaml.safe_load(f)


# fix every random source we use (python, numpy, torch) so reruns give the same splits and samples
def set_seed(seed: int) -> None:
    random.seed(seed)
    try:
        import numpy as np

        np.random.seed(seed)
    except ImportError:
        pass
    try:
        import torch

        torch.manual_seed(seed)
    except ImportError:
        pass


# one JSON object per line -> list of dicts (blank lines skipped)
def read_jsonl(path) -> list[dict]:
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


# opposite of read_jsonl; creates the folder if it doesn't exist yet
def write_jsonl(path, rows) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


# save a single object (stats, reports) as pretty-printed JSON
def write_json(path, obj) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


# ------------------------------------------------------------------------ prompts
# system prompt when documents are given. the 'say so if the docs don't have it' line matters:
# RAFT trains with missing gold passages, so the model should learn to admit it
SYSTEM = (
    "You are a customer-support agent. Use the reference documents to answer the "
    "customer's last message. Be accurate and concise, and do not invent policies "
    "or numbers. If the documents do not contain the answer, say so."
)
# closed-book version, used when there are no documents in the prompt
SYSTEM_CLOSED = "You are a customer-support agent. Answer the customer's last message accurately and concisely."
# extra instruction for the cot style: quote the evidence first, then answer
COT_INSTR = (
    "First copy the sentences you rely on between ##begin_quote## and ##end_quote##, "
    "then write the reply after ##Answer:"
)


# turn the conversation into the search query used by the retriever
def retrieval_query(history: list[dict]) -> str:
    """Dialogue-aware query: the last two customer turns."""
    # a follow-up like "how do I do that?" means nothing on its own, so the previous customer turn comes along
    users = [t["text"] for t in history if t["role"] == "user"]
    return " ".join(users[-2:])


# one passage as the model sees it: [n] title - section, then the text
def fmt_passage(i: int, p: dict) -> str:
    head = p["title"] + (f" - {p['section']}" if p.get("section") else "")
    return f"[{i}] {head}\n{p['text']}"


# build the chat prompt: documents (if any) + conversation so far + what to write.
# used for training AND evaluation, so RAFT is tested on exactly the format it learned
def build_messages(history: list[dict], passages: list[dict], style: str = "direct") -> list[dict]:
    parts = []
    if passages:
        parts.append("Reference documents:\n" + "\n\n".join(fmt_passage(i + 1, p) for i, p in enumerate(passages)))
    conv = "\n".join(f"{'Customer' if t['role'] == 'user' else 'Agent'}: {t['text']}" for t in history)
    parts.append("Conversation so far:\n" + conv)
    parts.append("Write the agent's next reply." + (" " + COT_INSTR if style == "cot" else ""))
    return [
        {"role": "system", "content": SYSTEM if passages else SYSTEM_CLOSED},
        {"role": "user", "content": "\n\n".join(parts)},
    ]


# the text the model is trained to produce: quote + answer (cot) or just the answer (direct)
def build_target(style: str, quote: str, response: str) -> str:
    # doc2dial already gives the grounding span, so the quote half of the target is free (no teacher model)
    if style == "cot":
        return f"##Reason: The relevant text says ##begin_quote## {quote} ##end_quote##.\n##Answer: {response}"
    return response


# grabs whatever sits between the quote markers
_QUOTE_RE = re.compile(r"##begin_quote##(.*?)##end_quote##", re.S)


# split a model output into the final answer and its quotes - only the answer gets scored
def parse_output(text: str) -> tuple[str, list[str]]:
    """Return (final answer, quoted spans). Works for direct and cot outputs."""
    quotes = [q.strip() for q in _QUOTE_RE.findall(text)]
    # base models never write ##Answer:, so fall back to the whole text minus the markers
    if "##Answer:" in text:
        ans = text.rsplit("##Answer:", 1)[1]
    else:
        ans = _QUOTE_RE.sub("", text).replace("##Reason:", "")
    return ans.strip(), quotes


# ------------------------------------------------------------------------ metrics
# articles and stop words are ignored when comparing texts
_ARTICLES = re.compile(r"\b(a|an|the)\b")
_STOP = set(
    "a an the and or but if of to in on for with at by from as is are was were be been it this that these those "
    "you your i we our they their he she his her not no do does did can will would should could may might have has had "
    "so than then there here what which who whom when where how about into over under up out".split()
)


# clean a string before comparing, so 'The SSA.' and 'ssa' count as the same
def normalize(s: str) -> str:
    # SQuAD-style: lowercase, drop punctuation and articles
    s = s.lower()
    s = "".join(ch for ch in s if ch not in set(string.punctuation))
    s = _ARTICLES.sub(" ", s)
    return " ".join(s.split())


# word-overlap F1 between our reply and the human agent's reply (0 = nothing shared, 1 = same words)
def token_f1(pred: str, gold: str) -> float:
    p, g = normalize(pred).split(), normalize(gold).split()
    if not p or not g:
        return float(p == g)
    common = Counter(p) & Counter(g)
    n = sum(common.values())
    if n == 0:
        return 0.0
    prec, rec = n / len(p), n / len(g)
    return 2 * prec * rec / (prec + rec)


# like F1 but based on the longest common subsequence, so word order counts a bit
def rouge_l(pred: str, gold: str) -> float:
    # plain LCS DP; the 200-token cap keeps it cheap when a model rambles
    p, g = normalize(pred).split()[:200], normalize(gold).split()[:200]
    if not p or not g:
        return float(p == g)
    prev = [0] * (len(g) + 1)
    for a in p:
        cur = [0]
        for j, b in enumerate(g):
            cur.append(prev[j] + 1 if a == b else max(prev[j + 1], cur[j]))
        prev = cur
    lcs = prev[-1]
    if lcs == 0:
        return 0.0
    prec, rec = lcs / len(p), lcs / len(g)
    return 2 * prec * rec / (prec + rec)


# how much of the answer is backed by the documents we passed in - low value = likely made up
def groundedness(answer: str, context: str) -> float:
    """Share of content words in the answer that appear in the supplied context (hallucination proxy)."""
    # crude on purpose: word overlap, not entailment. good for comparing models side by side,
    # not as an absolute hallucination rate
    a = [w for w in normalize(answer).split() if w not in _STOP]
    if not a:
        return 1.0
    c = set(normalize(context).split())
    return sum(w in c for w in a) / len(a)


# did the model quote text that really exists in the context, or invent it?
def quote_grounded(quotes: list[str], context: str) -> float | None:
    """Share of quoted spans that really occur in the context (None if no quotes)."""
    # only the cot model quotes - this checks it isn't inventing them
    if not quotes:
        return None
    c = normalize(context)
    return sum(normalize(q) in c for q in quotes if normalize(q)) / max(1, sum(1 for q in quotes if normalize(q)))


# p50 / p95 for latency numbers (linear interpolation, same idea as numpy.percentile)
def percentile(xs: list[float], q: float) -> float:
    xs = sorted(xs)
    if not xs:
        return float("nan")
    k = (len(xs) - 1) * q / 100
    lo, hi = int(k), min(int(k) + 1, len(xs) - 1)
    return xs[lo] + (xs[hi] - xs[lo]) * (k - lo)


# ---------------------------------------------------------------------- retriever
# dense retriever: embed all passages once, then a query is one matrix multiply + top-k.
# the exact same object is used for building RAFT data, RAG eval and the latency benchmark
class Retriever:
    """Dense retriever (bge-small by default). Whole corpus is a few thousand passages -> brute force matmul."""

    # load the embedding model and embed the corpus (or load the cached embeddings)
    def __init__(self, passages: list[dict], embed_model: str, prefix: str, device: str | None = None, emb_path=None):
        import numpy as np
        import torch
        from sentence_transformers import SentenceTransformer

        self.torch = torch
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.prefix = prefix
        self.passages = passages
        self.model = SentenceTransformer(embed_model, device=self.device)
        if self.device == "cuda":
            self.model.half()
        # embeddings are cached to disk so eval / bench don't re-embed the corpus every time
        if emb_path and Path(emb_path).exists():
            emb = torch.from_numpy(np.load(emb_path))
        else:
            texts = [f"{p['title']} - {p['section']}\n{p['text']}" for p in passages]
            emb = self.model.encode(texts, batch_size=64, normalize_embeddings=True, convert_to_tensor=True, show_progress_bar=True).float().cpu()
            if emb_path:
                Path(emb_path).parent.mkdir(parents=True, exist_ok=True)
                np.save(emb_path, emb.numpy())
        self.emb = emb.to(self.device).half() if self.device == "cuda" else emb

    # top-k passage indices + similarity scores for many queries, in batches of `bs`
    def search_batch(self, queries: list[str], k: int, bs: int = 128) -> tuple[list[list[int]], list[list[float]]]:
        idxs, scores = [], []
        for i in range(0, len(queries), bs):
            q = self.model.encode([self.prefix + x for x in queries[i : i + bs]], normalize_embeddings=True, convert_to_tensor=True, show_progress_bar=False)
            sims = q.to(self.emb.dtype) @ self.emb.T
            s, ix = sims.topk(min(k, sims.shape[1]), dim=1)
            idxs += ix.cpu().tolist()
            scores += s.float().cpu().tolist()
        return idxs, scores

    # single-query version, used in the benchmark where queries arrive one by one
    def search(self, query: str, k: int) -> list[int]:
        return self.search_batch([query], k)[0][0]

In [ ]:
%%writefile /kaggle/working/proj/src/prepare_data.py
"""Stage 1 - download Doc2Dial and turn it into: passages (the RAG corpus) + QA examples.

Each example = one agent turn that is grounded in the documents:
  history (previous turns) -> gold agent reply, gold passage ids, gold quote (the grounding spans).

Splits (all written to --out_dir):
  train / valid      dialogues from doc2dial train on *seen* docs
  test_seen          doc2dial validation dialogues on seen docs   (same corpus RAFT trained on)
  test_unseen        dialogues on held-out docs never used in training (tests generalisation)

    python src/prepare_data.py --out_dir work/data
    python src/prepare_data.py --raw_dir /kaggle/input/doc2dial-raw --out_dir work/data   # offline fallback
"""
from __future__ import annotations

import argparse
import random
import urllib.request
import zipfile
from pathlib import Path

from common import load_cfg, read_jsonl, retrieval_query, set_seed, write_json, write_jsonl  # noqa: F401
import json


# download the Doc2Dial zip once (skipped if already there) and unpack it
def download_and_extract(url: str, dest: Path) -> None:
    dest.mkdir(parents=True, exist_ok=True)
    zpath = dest / "doc2dial.zip"
    if not zpath.exists():
        req = urllib.request.Request(url, headers={"User-Agent": "raft-vs-rag/1.0"})
        with urllib.request.urlopen(req, timeout=120) as r, open(zpath, "wb") as f:
            f.write(r.read())
    root = dest.resolve()
    with zipfile.ZipFile(zpath) as zf:
        for m in zf.namelist():  # zip-slip check before extracting
            if not (root / m).resolve().is_relative_to(root):
                raise RuntimeError(f"unsafe path in zip: {m}")
        zf.extractall(dest)


# find a file anywhere under root - the zip's folder layout isn't something to rely on
def find_one(root: Path, name: str) -> Path:
    hits = sorted(root.rglob(name))
    if not hits:
        raise FileNotFoundError(f"{name} not found under {root}")
    return hits[0]


# the json files wrap everything in one top-level key ('doc_data' / 'dial_data'); strip it if present
def unwrap(obj: dict, key: str) -> dict:
    return obj.get(key, obj)


# int() that doesn't crash on missing or odd values - only used for sorting spans
def _num(x, default=0) -> int:
    try:
        return int(x)
    except (TypeError, ValueError):
        return default


# titles/ids can carry a '#...' suffix (e.g. 'Some Title#3_0'); keep just the readable part
def clean_title(t: str) -> str:
    return (t or "").split("#")[0].strip()


# turn every document into ~max_words passages that never cross a section boundary
def build_passages(doc_data: dict, max_words: int) -> tuple[list[dict], dict]:
    """Chunk each document by section. Returns (passages, {(doc_id, sp_id): passage_id})."""
    # doc2dial spans are sentence-sized, too small to retrieve on their own, so they get stitched back
    # together within a section up to ~max_words. sp2pid remembers which passage each span ended up in,
    # which is how the gold labels survive the chunking.
    passages, sp2pid = [], {}
    for domain, docs in doc_data.items():
        for doc_id, doc in docs.items():
            spans = doc.get("spans") or {}
            items = sorted(spans.items(), key=lambda kv: _num(kv[1].get("start_sp")))
            cur, n = None, 0

            # close the passage being built and start fresh
            def flush():
                nonlocal cur, n
                if cur and cur["words"] > 0:
                    cur["text"] = " ".join(cur.pop("_t"))
                    passages.append(cur)
                cur = None

            for sp_key, sp in items:
                text = " ".join((sp.get("text_sp") or "").split())
                if not text:
                    continue
                sec = str(sp.get("id_sec", ""))
                if cur is None or sec != cur["_sec"] or cur["words"] >= max_words:
                    flush()
                    pid = f"{doc_id}::{n}"
                    n += 1
                    cur = {"pid": pid, "doc_id": doc_id, "domain": domain, "title": clean_title(doc.get("title", doc_id)),
                           "section": " ".join((sp.get("title") or "").split()), "_sec": sec, "words": 0, "_t": []}
                cur["_t"].append(text)
                cur["words"] += len(text.split())
                sp2pid[(doc_id, str(sp_key))] = cur["pid"]
            flush()
    for p in passages:
        p.pop("_sec", None)
    return passages, sp2pid


# one example per grounded agent turn: recent history -> human reply + which passages it came from
def build_examples(dial_data: dict, doc_data: dict, sp2pid: dict, hist_turns: int, source: str) -> list[dict]:
    span_text = {(d, str(k)): " ".join((s.get("text_sp") or "").split())
                 for dom in doc_data.values() for d, doc in dom.items() for k, s in (doc.get("spans") or {}).items()}
    out = []
    for domain, docs in dial_data.items():
        for doc_id, dials in docs.items():
            for dial in dials:
                turns = dial.get("turns", [])
                did = dial.get("dial_id", "")
                for i, t in enumerate(turns):
                    # only agent turns that answer a customer turn
                    if t.get("role") != "agent" or i == 0 or turns[i - 1].get("role") != "user":
                        continue
                    resp = " ".join((t.get("utterance") or "").split())
                    # skip "ok" / "thanks" style replies and the odd very long one
                    if not (3 <= len(resp.split()) <= 120):
                        continue
                    refs = [str(r.get("sp_id")) for r in (t.get("references") or []) if r.get("sp_id") is not None]
                    gold = list(dict.fromkeys(sp2pid[(doc_id, r)] for r in refs if (doc_id, r) in sp2pid))
                    # no grounding span -> nothing to score retrieval or grounding against
                    if not gold:
                        continue
                    quote = " ".join(dict.fromkeys(span_text.get((doc_id, r), "") for r in refs if span_text.get((doc_id, r))))
                    quote = " ".join(quote.split()[:70])
                    hist = [{"role": x["role"], "text": " ".join((x.get("utterance") or "").split())}
                            for x in turns[max(0, i - hist_turns) : i]]
                    out.append({"id": f"{did}_{t.get('turn_id', i)}", "dial_id": did, "doc_id": doc_id, "domain": domain,
                                "history": hist, "response": resp, "gold_pids": gold, "gold_quote": quote,
                                "retrieval_query": retrieval_query(hist), "source": source})
    return out


# download -> chunk -> build examples -> split (holding out whole documents) -> write files + stats
def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default="configs/config.yaml")
    ap.add_argument("--out_dir", default="work/data")
    ap.add_argument("--raw_dir", default=None, help="folder that already contains doc2dial_*.json (skips download)")
    a = ap.parse_args()
    cfg = load_cfg(a.config)
    d = cfg["data"]
    set_seed(cfg["seed"])
    rng = random.Random(cfg["seed"])
    out = Path(a.out_dir)

    raw = Path(a.raw_dir) if a.raw_dir else out.parent / "raw"
    if not a.raw_dir:
        download_and_extract(d["doc2dial_url"], raw)
    # raw doc2dial: the documents, plus train and validation dialogues
    doc_data = unwrap(json.load(open(find_one(raw, "doc2dial_doc.json"))), "doc_data")
    tr_dial = unwrap(json.load(open(find_one(raw, "doc2dial_dial_train.json"))), "dial_data")
    va_dial = unwrap(json.load(open(find_one(raw, "doc2dial_dial_validation.json"))), "dial_data")

    passages, sp2pid = build_passages(doc_data, d["passage_words"])
    all_docs = sorted({p["doc_id"] for p in passages})
    # held out by document, not by dialogue - otherwise "unseen" would still share docs with train
    holdout = set(rng.sample(all_docs, max(1, int(len(all_docs) * d["holdout_doc_frac"]))))
    write_jsonl(out / "passages.jsonl", passages)

    # turn dialogues into examples, then sort them into pools by whether their doc is held out
    tr = build_examples(tr_dial, doc_data, sp2pid, d["history_turns"], "train")
    va = build_examples(va_dial, doc_data, sp2pid, d["history_turns"], "validation")

    unseen_pool = [e for e in tr + va if e["doc_id"] in holdout]
    train_pool = [e for e in tr if e["doc_id"] not in holdout]
    seen_pool = [e for e in va if e["doc_id"] not in holdout]

    # valid is split by dialogue so turns of one conversation don't land on both sides
    dials = sorted({e["dial_id"] for e in train_pool})
    valid_d = set(rng.sample(dials, max(1, int(len(dials) * d["valid_dialogue_frac"]))))
    valid_pool = [e for e in train_pool if e["dial_id"] in valid_d]
    train_pool = [e for e in train_pool if e["dial_id"] not in valid_d]

    # random subset of at most n rows, so split sizes stay under the config limits
    def cap(rows, n):
        rows = rows[:]
        rng.shuffle(rows)
        return rows[:n]

    splits = {"train": cap(train_pool, d["max_train"]), "valid": cap(valid_pool, d["max_valid"]),
              "test_seen": cap(seen_pool, d["max_test_seen"]), "test_unseen": cap(unseen_pool, d["max_test_unseen"])}
    for name, rows in splits.items():
        write_jsonl(out / f"{name}.jsonl", rows)

    stats = {"n_docs": len(all_docs), "n_holdout_docs": len(holdout), "n_passages": len(passages),
             "avg_passage_words": round(sum(p["words"] for p in passages) / max(1, len(passages)), 1),
             **{f"n_{k}": len(v) for k, v in splits.items()},
             "pool_sizes": {"train": len(train_pool), "seen": len(seen_pool), "unseen": len(unseen_pool)},
             "domains": {dom: sum(p["domain"] == dom for p in passages) for dom in {p["domain"] for p in passages}}}
    write_json(out / "stats.json", stats)
    print(json.dumps(stats, indent=2))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/proj/src/build_index.py
"""Stage 2 - embed the corpus, retrieve top-N passages for every query, report retrieval quality.

Writes  ret_<split>.jsonl  ({id, retrieved: [pid...], scores}) and retrieval_report.json
The same retrieval results feed (a) hard negatives for RAFT training and (b) the RAG baseline.

    python src/build_index.py --data_dir work/data
"""
from __future__ import annotations

import argparse
from pathlib import Path

from common import Retriever, load_cfg, read_jsonl, write_json, write_jsonl

# every split gets retrieved: train/valid for RAFT distractors, the test splits for RAG eval
SPLITS = ["train", "valid", "test_seen", "test_unseen"]


# embed the corpus, retrieve top-N for every example, and measure how often the gold passage shows up
def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default="configs/config.yaml")
    ap.add_argument("--data_dir", default="work/data")
    a = ap.parse_args()
    cfg = load_cfg(a.config)
    rc = cfg["retrieval"]
    dd = Path(a.data_dir)

    passages = read_jsonl(dd / "passages.jsonl")
    pids = [p["pid"] for p in passages]
    pid_doc = {p["pid"]: p["doc_id"] for p in passages}
    ret = Retriever(passages, rc["embed_model"], rc["query_prefix"], emb_path=dd / "passage_emb.npy")

    report = {}
    for split in SPLITS:
        rows = read_jsonl(dd / f"{split}.jsonl")
        idxs, scores = ret.search_batch([r["retrieval_query"] for r in rows], rc["save_top"])
        # counters for recall@k (gold found in top k), MRR (how high the first gold hit ranks), same-doc rate
        out, hits, mrr, dochit = [], {1: 0, 2: 0, 4: 0, 8: 0, 20: 0}, 0.0, 0
        for r, ix, sc in zip(rows, idxs, scores):
            got = [pids[i] for i in ix]
            out.append({"id": r["id"], "retrieved": got, "scores": [round(s, 4) for s in sc]})
            gold = set(r["gold_pids"])
            for k in hits:
                hits[k] += bool(gold & set(got[:k]))
            rank = next((j + 1 for j, p in enumerate(got) if p in gold), None)
            mrr += 1 / rank if rank else 0
            # even when the exact passage is missed, is the top hit at least from the right document?
            dochit += pid_doc[got[0]] == r["doc_id"]
        write_jsonl(dd / f"ret_{split}.jsonl", out)
        n = max(1, len(rows))
        report[split] = {"n": len(rows), **{f"recall@{k}": round(v / n, 4) for k, v in hits.items()},
                         "mrr": round(mrr / n, 4), "top1_same_doc": round(dochit / n, 4)}
        print(split, report[split])
    write_json(dd / "retrieval_report.json", report)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /kaggle/working/proj/src/build_raft.py
"""Stage 3 - build the RAFT fine-tuning set (Zhang et al., 2024).

For each training question:
  * with prob p_oracle the context holds the gold passage(s) + distractors,
    otherwise distractors only (teaches the model not to trust irrelevant docs / to rely on learned domain knowledge)
  * distractors are mostly retriever hard-negatives (what RAG would really feed the model) + some random ones
  * target = quote of the grounding span(s), then the human agent's reply   (style=cot)
            or just the reply                                             (style=direct)
No teacher LLM is needed: Doc2Dial already gives gold replies AND gold grounding spans.

    python src/build_raft.py --data_dir work/data --style cot
"""
from __future__ import annotations

import argparse
import random
from pathlib import Path

from common import build_messages, build_target, load_cfg, read_jsonl, set_seed, write_json, write_jsonl


# pick the n_docs passages for one training prompt: maybe the gold one(s), the rest distractors.
# hard distractors come from the retriever, so they look like what RAG would really return
def make_context(ex: dict, retrieved: list[str], all_pids: list[str], n_docs: int, p_oracle: float, hard_frac: float, rng: random.Random):
    gold = ex["gold_pids"][:2]  # at most 2, so distractors still make up most of the context
    # the other (1 - p_oracle) of the time there is no gold passage at all. that's the part of RAFT that
    # teaches the model not to blindly trust whatever it's handed
    use_oracle = rng.random() < p_oracle
    ctx = list(gold) if use_oracle else []
    hard = [p for p in retrieved[:10] if p not in gold]
    rng.shuffle(hard)
    tries = 0
    while len(ctx) < n_docs and tries < 200:  # tries cap = safety net against an endless loop
        tries += 1
        p = hard.pop() if (hard and rng.random() < hard_frac) else rng.choice(all_pids)
        if p not in gold and p not in ctx:
            ctx.append(p)
    ctx = ctx[:n_docs]
    rng.shuffle(ctx)  # so the gold passage isn't always in slot 1
    return ctx, use_oracle


# build prompt/completion pairs for train + valid, drop the ones that are too long, save stats
def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default="configs/config.yaml")
    ap.add_argument("--data_dir", default="work/data")
    ap.add_argument("--out_dir", default=None, help="default: <data_dir>/raft_<style>")
    ap.add_argument("--style", default=None, choices=["cot", "direct"])
    ap.add_argument("--p_oracle", type=float, default=None)
    ap.add_argument("--no_docs", action="store_true", help="closed-book SFT ablation (no documents in the prompt)")
    a = ap.parse_args()
    cfg = load_cfg(a.config)
    rf = cfg["raft"]
    style = a.style or rf["style"]
    p_oracle = rf["p_oracle"] if a.p_oracle is None else a.p_oracle
    set_seed(cfg["seed"])
    rng = random.Random(cfg["seed"])
    dd = Path(a.data_dir)
    out = Path(a.out_dir) if a.out_dir else dd / (f"sft_closedbook_{style}" if a.no_docs else f"raft_{style}")

    passages = read_jsonl(dd / "passages.jsonl")
    by_pid = {p["pid"]: p for p in passages}
    all_pids = list(by_pid)

    tok = None
    try:
        from transformers import AutoTokenizer

        tok = AutoTokenizer.from_pretrained(cfg["train"]["base_model"])
    except Exception as e:  # no HF access -> rough length estimate is good enough for filtering
        print(f"[warn] tokenizer unavailable ({e}); using a words*1.6 length estimate")

    # length of prompt + target in tokens, with the same chat template the trainer will use
    def n_tokens(msgs) -> int:
        if tok:
            return len(tok.apply_chat_template(msgs, tokenize=True, add_generation_prompt=False))
        return int(sum(len(m["content"].split()) for m in msgs) * 1.6)

    stats = {}
    for split in ["train", "valid"]:
        rows = read_jsonl(dd / f"{split}.jsonl")
        ret = {r["id"]: r["retrieved"] for r in read_jsonl(dd / f"ret_{split}.jsonl")}
        sft, meta, dropped, oracle_n = [], [], 0, 0
        # for each example: choose context -> build prompt -> build target -> keep it if it fits
        for ex in rows:
            if a.no_docs:
                ctx, used = [], False
            else:
                ctx, used = make_context(ex, ret[ex["id"]], all_pids, rf["n_docs"], p_oracle, rf["hard_neg_frac"], rng)
            prompt = build_messages(ex["history"], [by_pid[p] for p in ctx], style)
            completion = [{"role": "assistant", "content": build_target(style, ex["gold_quote"], ex["response"])}]
            if n_tokens(prompt + completion) > rf["max_tokens"]:
                dropped += 1
                continue
            oracle_n += used
            sft.append({"prompt": prompt, "completion": completion})  # TRL prompt-completion format
            meta.append({"id": ex["id"], "context_pids": ctx, "oracle_in_context": used})
        write_jsonl(out / f"{split}.jsonl", sft)
        write_jsonl(out / f"{split}_meta.jsonl", meta)
        stats[split] = {"kept": len(sft), "dropped_too_long": dropped, "oracle_in_context": oracle_n}
    stats["config"] = {"style": style, "p_oracle": p_oracle, "n_docs": 0 if a.no_docs else rf["n_docs"], "hard_neg_frac": rf["hard_neg_frac"]}
    write_json(out / "stats.json", stats)
    print(stats)


if __name__ == "__main__":
    main()

### Step 1 - download + build splits
If Kaggle can't reach `doc2dial.github.io`: download `doc2dial_v1.0.1.zip` locally, upload the three `doc2dial_*.json` files as a Kaggle dataset, add it as an input and use the commented `--raw_dir` line instead.

In [ ]:
!python src/prepare_data.py --out_dir work/data
# !python src/prepare_data.py --raw_dir /kaggle/input/<your-doc2dial-dataset> --out_dir work/data

### Step 2 - embed the corpus + retrieve
Prints recall@k per split (how often the gold passage is in the top k). Worth a look before training: if retrieval is weak, RAG never had a chance and the comparison mostly measures the retriever.

In [ ]:
!python src/build_index.py --data_dir work/data

### Step 3 - build the RAFT set
`cot` (quote, then answer) is the main one. `direct` (answer only) is just for an ablation.

In [ ]:
!python src/build_raft.py --data_dir work/data --style cot
!python src/build_raft.py --data_dir work/data --style direct   # optional ablation

### Sanity check
Print one full training example and make sure the prompt and target look right before spending GPU hours on them.

In [ ]:
import json
r = json.loads(open('work/data/raft_cot/train.jsonl').readline())
print(r['prompt'][0]['content'], '\n---\n', r['prompt'][1]['content'][:2500], '\n--- TARGET ---\n', r['completion'][0]['content'])
print(open('work/data/raft_cot/stats.json').read())

In [ ]:
!rm -f work/raw/doc2dial.zip && rm -rf work/raw   # keep the output small
!du -sh work/data